In [6]:
import rasterio
from rasterio.windows import from_bounds
from pathlib import Path

in_folder = Path(r"C:\Users\Carl\Desktop\GAEZ_V5_0120\Crop rasters")
out_folder = Path(r"C:\Users\Carl\Desktop\GAEZ_V5_0120\Clipped rasters")
out_folder.mkdir(parents=True, exist_ok=True)

west, south, east, north = 26, -5, 36, 5

rasters = [
    f for f in in_folder.iterdir()
    if f.is_file() and ".LRLM" in f.name.upper()
]

for in_raster in sorted(rasters):
    parts = in_raster.name.split(".")

    # Find LRLM, then take the part immediately before it
    parts_upper = [p.upper() for p in parts]
    lrw_index = parts_upper.index("LRLM")
    crop_name = parts[lrw_index - 1]

    out_raster = out_folder / f"{crop_name}.tif"

    print("Clipping", crop_name, "from", in_raster.name)

    with rasterio.open(in_raster) as src:
        window = from_bounds(west, south, east, north, src.transform)

        data = src.read(window=window)
        transform = src.window_transform(window)

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": data.shape[1],
            "width": data.shape[2],
            "transform": transform,
            "compress": "lzw"
        })

        with rasterio.open(out_raster, "w", **meta) as dst:
            dst.write(data)

print("Done.")

Clipping ALF from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.ALF.LRLM.tif
Clipping BAN from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.BAN.LRLM.tif
Clipping BCK from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.BCK.LRLM.tif
Clipping BRL from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.BRL.LRLM.tif
Clipping BSG from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.BSG.LRLM.tif
Clipping CAB from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.CAB.LRLM.tif
Clipping CAR from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.CAR.LRLM.tif
Clipping CHK from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.CHK.LRLM.tif
Clipping CIT from DATA_GAEZ-V5_MAPSET_RES05-SIX30AS_GAEZ-V5.RES05-SIX30AS.HP0120.AGERA5.HIST.CIT.LRLM.tif
Clipping COC from DATA_GAEZ-V5_MAPSET_RES05-SI

In [8]:
# stack rasters and save band-name list

import rasterio
from pathlib import Path

in_folder = Path(r"C:\Users\Carl\Desktop\GAEZ_V5_0120\Clipped rasters")
out_raster = in_folder / "GAEZ_V5_IX30AS_stacked.tif"
out_band_list = in_folder / "GAEZ_V5_IX30AS_band_names.txt"

crop_files = sorted([
    f for f in in_folder.glob("*.tif")
    if f.name.lower() != "gaez_v5_ix30as_stacked.tif"
])

print("Files to stack:", len(crop_files))

crop_names = [f.stem for f in crop_files]

print("Band order:")
for i, name in enumerate(crop_names, start=1):
    print(i, name)

with rasterio.open(crop_files[0]) as first:
    meta = first.meta.copy()

meta.update({
    "count": len(crop_files),
    "compress": "lzw"
})

with rasterio.open(out_raster, "w", **meta) as dst:
    for band_number, crop_file in enumerate(crop_files, start=1):
        crop_name = crop_file.stem

        print("Adding band", band_number, crop_name)

        with rasterio.open(crop_file) as src:
            data = src.read(1)
            dst.write(data, band_number)
            dst.set_band_description(band_number, crop_name)

# Save band names as an Earth Engine-ready list
with open(out_band_list, "w") as f:
    f.write("var cropNames = [\n")
    for name in crop_names:
        f.write(f'  "{name}",\n')
    f.write("];\n")

print("Done:", out_raster)
print("Band name list saved:", out_band_list)

Files to stack: 64
Band order:
1 ALF
2 BAN
3 BCK
4 BRL
5 BSG
6 CAB
7 CAR
8 CHK
9 CIT
10 COC
11 COF
12 CON
13 COT
14 COW
15 CSH
16 CST
17 CSV
18 FIML
19 FLX
20 FML
21 FONIO
22 GRD
23 GRM
24 JTR
25 MIS
26 MNG
27 MZE
28 MZS
29 NAP
30 OAT
31 OKRA
32 OLP
33 OLV
34 ONI
35 PEA
36 PHB
37 PIG
38 PML
39 PST
40 RCD
41 RCG
42 RCW
43 RSD
44 RUB
45 RYE
46 SES
47 SFL
48 SOY
49 SPO
50 SRG
51 SUB
52 SUC
53 SWG
54 TANN
55 TAROD
56 TAROW
57 TEA
58 TEF
59 TOB
60 TOM
61 WHE
62 WMEL
63 WPO
64 YAM
Adding band 1 ALF
Adding band 2 BAN
Adding band 3 BCK
Adding band 4 BRL
Adding band 5 BSG
Adding band 6 CAB
Adding band 7 CAR
Adding band 8 CHK
Adding band 9 CIT
Adding band 10 COC
Adding band 11 COF
Adding band 12 CON
Adding band 13 COT
Adding band 14 COW
Adding band 15 CSH
Adding band 16 CST
Adding band 17 CSV
Adding band 18 FIML
Adding band 19 FLX
Adding band 20 FML
Adding band 21 FONIO
Adding band 22 GRD
Adding band 23 GRM
Adding band 24 JTR
Adding band 25 MIS
Adding band 26 MNG
Adding band 27 MZE
Adding band 2

In [5]:
import rasterio
from pathlib import Path

raster = Path(r"C:\Users\Carl\Desktop\FAO SOILFER low input Rainfed\Clipped output\all_crops_stacked.tif")

with rasterio.open(raster) as src:
    print("File:", raster.name)
    print("Number of bands:", src.count)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

    print("\nDataset metadata:")
    print(src.tags())

    print("\nBand descriptions:")
    for i in range(1, src.count + 1):
        print(i, src.descriptions[i - 1])

    print("\nBand metadata:")
    for i in range(1, src.count + 1):
        print("Band", i, src.tags(i))

File: all_crops_stacked.tif
Number of bands: 23
CRS: EPSG:4326
Bounds: BoundingBox(left=26.0, bottom=-5.0, right=36.0, top=5.0)
Resolution: (0.008333333333333333, 0.008333333333333333)

Dataset metadata:
{'AREA_OR_POINT': 'Area'}

Band descriptions:
1 BAN
2 COW
3 CSH
4 CSV
5 FIML
6 FONIO
7 GRD
8 GRM
9 MZE
10 OKRA
11 PIG
12 PML
13 SES
14 SOY
15 SPO
16 SRG
17 TANN
18 TAROD
19 TAROW
20 TEF
21 TOM
22 WMEL
23 YAM

Band metadata:
Band 1 {}
Band 2 {}
Band 3 {}
Band 4 {}
Band 5 {}
Band 6 {}
Band 7 {}
Band 8 {}
Band 9 {}
Band 10 {}
Band 11 {}
Band 12 {}
Band 13 {}
Band 14 {}
Band 15 {}
Band 16 {}
Band 17 {}
Band 18 {}
Band 19 {}
Band 20 {}
Band 21 {}
Band 22 {}
Band 23 {}
